# Experiment 3: Wikipedia API Exploration

**Goal:** Test fetching music history articles from Wikipedia and evaluate text quality for NER/LLM-based triple extraction.

**Key questions:**
- How clean is the extracted article text?
- Can we split by section headers?
- How do we go from MusicBrainz/Discogs → Wikipedia article?
- What does the text look like for artist pages vs genre history pages?

**Rate limit:** ~200 req/sec, no auth needed.

## Setup

In [1]:
import requests
import json

WIKI_API = "https://en.wikipedia.org/w/api.php"
HEADERS = {
    "User-Agent": "KE-CW2-MusicHistory/0.1 (lucas@example.com)"
}

def wiki_get(params):
    """Helper for Wikipedia API requests."""
    params["format"] = "json"
    resp = requests.get(WIKI_API, params=params, headers=HEADERS)
    resp.raise_for_status()
    return resp.json()

def pp(data):
    print(json.dumps(data, indent=2, default=str))

## 1. Fetch an Artist Article — Plain Text

From Experiment 2, Discogs gave us `https://en.wikipedia.org/wiki/David_Bowie`.
The page title is `David_Bowie`. Let's fetch the full article text.

In [2]:
result = wiki_get({
    "action": "query",
    "titles": "David Bowie",
    "prop": "extracts",
    "explaintext": True,  # Plain text, no HTML
    "exlimit": 1
})

pages = result["query"]["pages"]
page = list(pages.values())[0]
text = page.get("extract", "")

print(f"Title: {page['title']}")
print(f"Page ID: {page['pageid']}")
print(f"Text length: {len(text)} characters")
print(f"\n{'='*60}")
print(text[:3000])
print(f"\n... [{len(text) - 3000} more characters]")

Title: David Bowie
Page ID: 8786
Text length: 85851 characters

David Robert Jones (8 January 1947 – 10 January 2016), known as David Bowie, was an English singer, songwriter and actor. Regarded as among the most influential musicians of the 20th century, Bowie received particular acclaim for his work in the 1970s. His career was marked by reinvention and visual presentation, and his music and stagecraft have had a significant impact on popular music.
Bowie studied art, music and design before embarking on a music career in 1962. He released a string of unsuccessful singles with local bands and a self-titled solo album (1967) before achieving his first top-five entry on the UK singles chart with "Space Oddity" (1969). After a period of experimentation, he re-emerged in 1972 during the glam rock era with the alter ego Ziggy Stardust. The single "Starman" and its album The Rise and Fall of Ziggy Stardust and the Spiders from Mars (1972) won him widespread popularity. In 1975, Bowie's sty

### 1a. Examine Section Structure

Wikipedia articles use `==` headers. Let's see what sections exist — this tells us what kinds of info we can extract.

In [3]:
# Get the section structure
sections_result = wiki_get({
    "action": "parse",
    "page": "David Bowie",
    "prop": "sections"
})

sections = sections_result["parse"]["sections"]
print(f"Sections ({len(sections)}):\n")
for s in sections:
    indent = "  " * (int(s['toclevel']) - 1)
    print(f"{indent}{s['number']}. {s['line']}")

Sections (44):

1. Early life
2. Music career
  2.1. 1962–1967: Early career to debut album
  2.2. 1968–1971: <i>Space Oddity</i> to <i>Hunky Dory</i>
  2.3. 1972–1974: Glam rock era
  2.4. 1974–1976: "Plastic soul" and the Thin White Duke
  2.5. 1976–1979: Berlin era
  2.6. 1980–1988: New Romantic and pop era
  2.7. 1989–1991: Tin Machine
  2.8. 1992–1998: Electronic period
  2.9. 1999–2012: Neoclassicist era
  2.10. 2013–2016: Final years
  2.11. Posthumous releases
3. Acting career
4. Other works
  4.1. Painter and art collector
  4.2. Writings
  4.3. Bowie Bonds
  4.4. Websites
  4.5. Philanthropy
5. Musicianship
6. Personal life
  6.1. Family
  6.2. Other relationships
    6.2.1. Coco Schwab
  6.3. Sexuality
  6.4. Spirituality and religion
  6.5. Political views
7. Death
8. Legacy
  8.1. Exhibitions
  8.2. Films
9. Awards and achievements
10. Commemoration
11. Discography
12. Selected filmography
13. Tours
14. See also
15. Notes
16. References
  16.1. Citations
  16.2. Bibliograp

### 1b. Fetch a Specific Section

We can fetch individual sections — useful for targeted extraction (e.g., just the "Musical style" or "Legacy" section).

In [4]:
# Fetch a specific section by index (e.g., section 0 = intro)
# Try finding a section about musical style, career, or discography
for s in sections:
    if any(kw in s['line'].lower() for kw in ['career', 'music', 'style', 'legacy', 'influence']):
        print(f"Interesting section: {s['number']}. {s['line']} (index: {s['index']})")

Interesting section: 2. Music career (index: 2)
Interesting section: 2.1. 1962–1967: Early career to debut album (index: 3)
Interesting section: 3. Acting career (index: 14)
Interesting section: 5. Musicianship (index: 21)
Interesting section: 8. Legacy (index: 30)


In [5]:
# Fetch a section by index — change the index based on what you found above
section_index = 0  # Change this to an interesting section index

section_result = wiki_get({
    "action": "query",
    "titles": "David Bowie",
    "prop": "extracts",
    "explaintext": True,
    "exsectionformat": "plain",
    "exlimit": 1
})

section_page = list(section_result["query"]["pages"].values())[0]
section_text = section_page.get("extract", "")
print(section_text[:2000])

David Robert Jones (8 January 1947 – 10 January 2016), known as David Bowie, was an English singer, songwriter and actor. Regarded as among the most influential musicians of the 20th century, Bowie received particular acclaim for his work in the 1970s. His career was marked by reinvention and visual presentation, and his music and stagecraft have had a significant impact on popular music.
Bowie studied art, music and design before embarking on a music career in 1962. He released a string of unsuccessful singles with local bands and a self-titled solo album (1967) before achieving his first top-five entry on the UK singles chart with "Space Oddity" (1969). After a period of experimentation, he re-emerged in 1972 during the glam rock era with the alter ego Ziggy Stardust. The single "Starman" and its album The Rise and Fall of Ziggy Stardust and the Spiders from Mars (1972) won him widespread popularity. In 1975, Bowie's style shifted towards a sound he characterised as "plastic soul", i

## 2. Fetch a Genre History Article

Artist pages give us info about individuals. Genre pages give us historical context, origins, and evolution — different types of triples.

In [6]:
# Try a few possible article titles for rock music history
test_titles = ["History of rock music", "Rock music", "Origins of rock and roll"]

for title in test_titles:
    result = wiki_get({
        "action": "query",
        "titles": title,
        "prop": "extracts",
        "explaintext": True,
        "exintro": True,
        "exlimit": 1
    })
    page = list(result["query"]["pages"].values())[0]
    extract = page.get("extract", "")
    print(f"'{title}' → {len(extract)} chars")
    if extract:
        print(f"  {extract[:300]}...\n")
    else:
        print(f"  (empty or redirect)\n")

'History of rock music' → 0 chars
  (empty or redirect)

'Rock music' → 3686 chars
  Rock music is a genre of popular music that originated in the United States as "rock and roll" in the late 1940s and early 1950s, developing into a range of styles from the mid-1960s, primarily in the United States and United Kingdom. It has its roots beginning as classic rock and roll, a style that...

'Origins of rock and roll' → 1847 chars
  The origins of rock and roll are complex.  Rock and roll emerged as a defined musical style in the United States in the early to mid-1950s. It derived most directly from the rhythm and blues music of the 1940s, which itself developed from earlier blues, the beat-heavy jump blues, boogie woogie, up-t...



In [7]:
# Sections for the genre article
genre_sections = wiki_get({
    "action": "parse",
    "page": "History of rock music",
    "prop": "sections"
})

print("Sections:\n")
for s in genre_sections["parse"]["sections"]:
    indent = "  " * (int(s['toclevel']) - 1)
    print(f"{indent}{s['number']}. {s['line']}")

Sections:



## 3. Fetch Categories

Wikipedia categories provide free classification — they can supplement our ontology.

In [8]:
cat_result = wiki_get({
    "action": "query",
    "titles": "David Bowie",
    "prop": "categories",
    "cllimit": 50
})

cat_page = list(cat_result["query"]["pages"].values())[0]
categories = cat_page.get("categories", [])

print(f"Categories ({len(categories)}):\n")
for cat in categories:
    print(f"  - {cat['title']}")

Categories (50):

  - Category:1947 births
  - Category:2016 deaths
  - Category:20th-century English LGBTQ people
  - Category:20th-century English male actors
  - Category:20th-century English male singers
  - Category:20th-century English singer-songwriters
  - Category:21st-century English LGBTQ people
  - Category:21st-century English male singers
  - Category:21st-century English singer-songwriters
  - Category:21st-century art collectors
  - Category:Actors from the London Borough of Bromley
  - Category:Actors from the London Borough of Lambeth
  - Category:All Wikipedia articles written in British English
  - Category:Androgynous people
  - Category:Art collectors from London
  - Category:Art pop musicians
  - Category:Art pop singers
  - Category:Art rock musicians
  - Category:Articles containing French-language text
  - Category:Articles with hAudio microformats
  - Category:Articles with hCards
  - Category:Articles with short description
  - Category:Bertelsmann Music Gro

## 4. Cross-Reference: Wikidata ID → Wikipedia Article

From Experiment 1, MusicBrainz gave us Wikidata ID `Q5383` for Bowie.
Let's go from Wikidata → Wikipedia article title.

In [9]:
# Wikidata API — get Wikipedia article from Wikidata ID
wikidata_id = "Q5383"  # David Bowie from MusicBrainz

wd_result = requests.get(
    f"https://www.wikidata.org/wiki/Special:EntityData/{wikidata_id}.json",
    headers=HEADERS
)

print(f"Status: {wd_result.status_code}")

if wd_result.status_code == 200:
    wd_data = wd_result.json()
    entity = wd_data["entities"][wikidata_id]

    # Get the English Wikipedia article title
    sitelinks = entity.get("sitelinks", {})
    enwiki = sitelinks.get("enwiki", {})
    wiki_title = enwiki.get("title", "Not found")

    print(f"Wikidata ID: {wikidata_id}")
    print(f"Wikipedia title: {wiki_title}")
    print(f"\nCross-reference: MusicBrainz → Wikidata ({wikidata_id}) → Wikipedia ('{wiki_title}') ✓")
else:
    print(f"Failed: {wd_result.status_code} — {wd_result.text[:200]}")
    print("\nFallback: use Discogs URL or search Wikipedia by artist name instead")

Status: 200
Wikidata ID: Q5383
Wikipedia title: David Bowie

Cross-reference: MusicBrainz → Wikidata (Q5383) → Wikipedia ('David Bowie') ✓


## 5. Alternative: Discogs URL → Wikipedia Title

From Experiment 2, Discogs URLs included `https://en.wikipedia.org/wiki/David_Bowie`.
We can also extract the title directly from that URL.

In [10]:
from urllib.parse import unquote

discogs_wiki_url = "https://en.wikipedia.org/wiki/David_Bowie"

# Extract title from URL
wiki_title_from_url = unquote(discogs_wiki_url.split("/wiki/")[-1].replace("_", " "))
print(f"Extracted title: '{wiki_title_from_url}'")

# Verify it works
verify = wiki_get({
    "action": "query",
    "titles": wiki_title_from_url,
    "prop": "info"
})

verify_page = list(verify["query"]["pages"].values())[0]
print(f"Page ID: {verify_page.get('pageid', 'NOT FOUND')}")
print(f"\nCross-reference: Discogs URLs → extract title → Wikipedia API ✓")

Extracted title: 'David Bowie'
Page ID: 8786

Cross-reference: Discogs URLs → extract title → Wikipedia API ✓


## 6. Batch Fetch — Multiple Music Articles

Test fetching several articles to see data consistency across different article types.

In [11]:
test_articles = [
    "The Beatles",
    "Rock music",
    "Grunge",
    "Abbey Road",
    "Grammy Award"
]

# Wikipedia allows up to 50 titles per request
batch_result = wiki_get({
    "action": "query",
    "titles": "|".join(test_articles),
    "prop": "extracts",
    "explaintext": True,
    "exintro": True,  # Just the intro paragraph
    "exlimit": len(test_articles)
})

for pid, page in batch_result["query"]["pages"].items():
    title = page.get("title", "?")
    extract = page.get("extract", "")
    print(f"\n{'='*60}")
    print(f"{title} ({len(extract)} chars)")
    print(f"{'='*60}")
    print(extract[:500])
    if len(extract) > 500:
        print(f"... [{len(extract)-500} more]")


Abbey Road (2085 chars)
Abbey Road is the eleventh studio album by the English rock band the Beatles, released on 26 September 1969, by Apple Records. It is the last album the group recorded, although Let It Be (1970) was the last album completed before the band's break-up in April 1970. It was mostly recorded in April, July, and August 1969, and topped the record charts in both the United States and the United Kingdom. A double A-side single from the album, "Something" / "Come Together", was released in October, which
... [1585 more]

Grammy Award (0 chars)


Grunge (2364 chars)
Grunge (originally known as the Seattle Sound) is an alternative rock genre and subculture that emerged during the mid-1980s in the U.S. state of Washington, particularly in Seattle and Olympia, and other nearby cities. Grunge fuses elements of punk rock and heavy metal, and features the distorted electric guitar sound used in both genres, as well as bass guitar, drums, and vocals. Grunge also incorporates in

## 7. Internal Links — Entity Discovery

Wikipedia's internal links are essentially entity references. We can use them for relationship discovery.

In [12]:
links_result = wiki_get({
    "action": "query",
    "titles": "David Bowie",
    "prop": "links",
    "pllimit": 100
})

links_page = list(links_result["query"]["pages"].values())[0]
links = links_page.get("links", [])

print(f"Internal links (first 100 of many):\n")
for link in links[:50]:
    print(f"  - {link['title']}")

Internal links (first 100 of many):

  - "Heroes" (David Bowie album)
  - "Heroes" (David Bowie song)
  - "Heroes" (album)
  - 'Tis a Pity She Was a Whore
  - (What's the Story) Morning Glory?
  - 100 Greatest Britons
  - 1984 (song)
  - 1984 MTV Video Music Awards
  - 1992 Los Angeles riots
  - 2014 Brit Awards
  - 2022 Cannes Film Festival
  - 20 Feet from Stardom
  - 21 (Adele album)
  - 25 (Adele album)
  - 30 (album)
  - 342843 Davidbowie
  - 500 Greatest Songs of All Time
  - A&E Networks
  - ABC-CLIO
  - AIR (French band)
  - AM (Arctic Monkeys album)
  - ARKive
  - A Brief Inquiry into Online Relationships
  - A Ghost Is Born
  - A Knight's Tale
  - A Million in Prizes: The Anthology
  - A New Career in a New Town (1977–1982)
  - A Reality Tour
  - A Reality Tour (album)
  - A Reality Tour (film)
  - A Rush of Blood to the Head
  - Aboriginal Australians
  - Absolute Beginners (David Bowie song)
  - Absolute Beginners (film)
  - Adam Clayton
  - Adam Wakeman
  - Adam and the An

## 8. Summary

| Feature | Quality | Notes |
|---|---|---|
| Article text | | Clean? Sections? Length? |
| Section structure | | What sections are available? |
| Categories | | Useful for classification? |
| Internal links | | Useful for entity discovery? |
| Cross-ref from Wikidata | | MusicBrainz → Wikidata → Wikipedia works? |
| Cross-ref from Discogs | | Discogs URL → Wikipedia title works? |
| Batch fetching | | Can fetch multiple articles at once? |
| Genre articles vs artist articles | | Different structure? Different extraction needs? |

Fill in your observations after running the cells above.

## Next Steps

- **Experiment 4**: Cross-reference all three sources for the same entity
- **Experiment 5**: Triple extraction from structured data (MusicBrainz/Discogs → RDF)
- **Experiment 6**: Triple extraction from text (Wikipedia → NER/LLM → triples)